# Notebook 01 | Market Impact Scaling and Calibration

Examine the impact assumptions used in the execution engine. The goal is to make the temporary, permanent, and transient impact components auditable before comparing full execution strategies going forward.

I want to focus on the following:
- participation-driven scaling of temporary impact
- relative magnitude of permanent impact
- transient impact decay under repeated trading pressure
- comparison between configured coefficients and a simple calibration anchor

## Base Scenario Setup

Load the base configuration and generate a representative synthetic intraday market path. The market path is used only to set realistic price and volume scales for the impact illustrations below.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from execution_engine.config import load_config
from execution_engine.impact.calibration import calibrate_temporary_coefficient
from execution_engine.impact.linear_impact import linear_temporary_impact
from execution_engine.impact.permanent_impact import linear_permanent_impact
from execution_engine.impact.square_root_impact import square_root_temporary_impact
from execution_engine.impact.transient_impact import TransientImpactModel
from execution_engine.market.price_process import IntradayMarketSimulator

config = load_config(PROJECT_ROOT / 'configs' / 'base.yaml')
market = IntradayMarketSimulator(config).simulate(seed=7)
market[['clock_time', 'base_mid_price', 'expected_volume', 'realized_volume', 'spread']].head()

## Temporary and Permanent Impact Scaling

The next figure compares linear temporary impact, square-root temporary impact, and linear permanent impact as a function of bucket participation. All curves are expressed in bps relative to the prevailing midprice.

In [ ]:
bucket_volume = market['expected_volume'].median()
price = market['base_mid_price'].iloc[0]
quantities = np.linspace(10_000, 400_000, 60)
participation = quantities / bucket_volume

linear_curve = [
    linear_temporary_impact(q, bucket_volume, price, config.impact.temporary_impact_coefficient)
    for q in quantities
]
sqrt_curve = [
    square_root_temporary_impact(
        q,
        bucket_volume,
        price,
        config.impact.temporary_impact_coefficient,
        exponent=config.impact.impact_exponent,
    )
    for q in quantities
]
permanent_curve = [
    linear_permanent_impact(q, config.market.adv, price, config.impact.permanent_impact_coefficient)
    for q in quantities
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(participation, np.array(linear_curve) * 10_000 / price, linewidth=2.0, label='Linear temporary impact')
ax.plot(participation, np.array(sqrt_curve) * 10_000 / price, linewidth=2.0, label='Square-root temporary impact')
ax.plot(participation, np.array(permanent_curve) * 10_000 / price, linewidth=2.0, label='Permanent impact')
ax.set_xlabel('Bucket Participation Rate')
ax.set_ylabel('Impact (bps vs. midprice)')
ax.set_title('Temporary and Permanent Impact by Participation Rate')
ax.grid(linestyle='--', alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Transient Footprint and Resilience

A repeated sequence of bucket participations is applied to the transient-impact model to illustrate how residual footprint decays when trading pressure subsides and rebuilds when participation accelerates again.

In [ ]:
transient = TransientImpactModel(
    coefficient=config.impact.temporary_impact_coefficient * config.impact.information_leakage_coefficient,
    exponent=config.impact.impact_exponent,
    decay=config.impact.resilience_decay,
)

participation_path = np.array([0.08, 0.10, 0.14, 0.12, 0.06, 0.00, 0.00, 0.04])
transient_costs = []
for rate in participation_path:
    quantity = rate * bucket_volume
    transient_costs.append(transient.evaluate(quantity, bucket_volume, price))

transient_frame = pd.DataFrame(
    {
        'bucket': range(len(transient_costs)),
        'participation_rate': participation_path,
        'transient_cost_bps': np.array(transient_costs) * 10_000 / price,
    }
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(transient_frame['bucket'], transient_frame['transient_cost_bps'], marker='o', linewidth=2.0, color='#7f5539')
ax.set_xlabel('Bucket')
ax.set_ylabel('Transient Cost (bps vs. midprice)')
ax.set_title('Transient Impact Response to Repeated Trading Pressure')
ax.grid(linestyle='--', alpha=0.25)
plt.tight_layout()
plt.show()

transient_frame

## Calibration Anchor Check

The configured temp impact coefficient is compared with a simple anchor: 12 bps of temporary impact at 10% participation under a square-root specification. This is not a full empirical calibration, but it is a useful reasonableness check.

In [ ]:
heuristic_coeff = calibrate_temporary_coefficient(reference_participation=0.10, reference_impact_bps=12.0, exponent=0.5)
comparison = pd.DataFrame(
    {
        'configured_temp_coefficient': [config.impact.temporary_impact_coefficient],
        'heuristic_temp_coefficient_10pct_12bps': [heuristic_coeff],
        'configured_to_anchor_ratio': [config.impact.temporary_impact_coefficient / heuristic_coeff],
    }
)
comparison